# Buổi 3 — ANOVA & Kiểm định phi tham số (Bài 8, 9)

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

import numpy as np, pandas as pd

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' dùng cho cả 6 buổi (không phải dữ liệu thật)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)      # giờ tự học / tuần
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")           # hứng thú (ẩn)
    anxiety = rng.normal(0, 1, n) - 0.25 * interest                        # lo âu (ẩn)
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    # "bẫy" cố ý cài vào dữ liệu để buổi 1 phát hiện:
    d.loc[6, "study_hours"] = 48.0          # gõ nhầm 4.8 thành 48
    d.loc[[11, 57, 130], "h2"] = np.nan     # thiếu dữ liệu
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print(df.shape)

In [ ]:
df.loc[df.study_hours > 20, "study_hours"] /= 10

## 1. ANOVA từ nguyên lý đầu tiên: chia phương sai thành 2 phần
Tổng biến thiên (SST) = biến thiên **giữa** nhóm (SSB) + biến thiên **trong** nhóm (SSW). F = (SSB/df1)/(SSW/df2).
SPSS: `Analyze > Compare Means > One-Way ANOVA` (chọn Post Hoc: Tukey; Options: Homogeneity, Welch).

In [ ]:
g = df.groupby("school").math
grand = df.math.mean()
SSB = sum(len(v) * (v.mean() - grand) ** 2 for _, v in g); SSW = sum(((v - v.mean()) ** 2).sum() for _, v in g)
k, N = df.school.nunique(), len(df); F = (SSB / (k - 1)) / (SSW / (N - k))
print(f"SSB={SSB:.0f} SSW={SSW:.0f}  F thủ công={F:.2f}  p={stats.f.sf(F, k-1, N-k):.3g}")
print("scipy:", stats.f_oneway(*[v for _, v in g]))
print("eta² (mức ảnh hưởng) =", round(SSB / (SSB + SSW), 3))

## 2. Kiểm tra giả định — nếu vi phạm thì sao?

In [ ]:
import statsmodels.formula.api as smf, statsmodels.api as sm
m = smf.ols("math ~ C(school)", df).fit()
print("Levene:", stats.levene(*[v for _, v in g]))
print("Shapiro trên phần dư:", stats.shapiro(m.resid))
sm.qqplot(m.resid, line="s"); plt.show()

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
print(pairwise_tukeyhsd(df.math, df.school))       # Post-hoc: cặp nào khác nhau?
print("Kruskal-Wallis (phi tham số):", stats.kruskal(*[v for _, v in g]))

**❓** Vì sao không chạy 3 lần t-test cho A–B, A–C, B–C mà phải dùng ANOVA + post-hoc? (gợi ý: xác suất ít nhất 1 lần dương tính giả = 1 − 0.95³ ≈ ?)

## 3. ANOVA hai chiều và tương tác
Phương pháp dạy có hiệu quả **như nhau ở mọi trường** không?

In [ ]:
df["gain"] = df.posttest - df.pretest
m2 = smf.ols("gain ~ C(method) * C(school)", df).fit()
print(sm.stats.anova_lm(m2, typ=2).round(3))
sns.pointplot(data=df, x="school", y="gain", hue="method", dodge=.2); plt.title("Đường không song song = có tương tác"); plt.show()

## 4. Phi tham số — khi dữ liệu không đủ 'đẹp'
SPSS: `Analyze > Nonparametric Tests > Independent Samples` / `Legacy Dialogs > Chi-square`.

In [ ]:
print("Mann-Whitney (h1 theo giới, Likert):", stats.mannwhitneyu(df[df.gender == "Nam"].h1, df[df.gender == "Nữ"].h1))
ct = pd.crosstab(df.gender, df.method); print(ct); chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f"Chi-square độc lập: χ²={chi2:.2f}, df={dof}, p={p:.3f}")
print("Wilcoxon bắt cặp pre-post:", stats.wilcoxon(df.posttest, df.pretest))

### Bảng chọn kiểm định (để in ra dán bàn)
| Câu hỏi | Tham số | Phi tham số |
|---|---|---|
| 2 nhóm độc lập | t-test Welch | Mann-Whitney U |
| 2 lần đo cùng đối tượng | t bắt cặp | Wilcoxon |
| ≥3 nhóm độc lập | ANOVA (+Tukey) | Kruskal-Wallis |
| 2 biến định danh | — | Chi-square |

## 5. Bài tập
1. Kiểm định `pretest` khác nhau giữa các trường không? Hai nhóm phương pháp có cân bằng nhau ở `pretest` không? Điều này nói gì về thiết kế nghiên cứu?
2. Cố ý thêm 5 giá trị ngoại lai vào `math` của trường C rồi chạy lại ANOVA và Kruskal. Kết quả nào bền hơn?